# Session1_Task2 — Data Cleaning & Transformation

In [1]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

c = pd.read_csv('customers.csv')
s = pd.read_csv('sales_transactions.csv')

In [2]:
# --- 1. Missing Values ---

# age: เติมด้วย median (ค่ากลาง)
# .fillna() → เติมค่าว่าง (NaN) ด้วยค่าที่กำหนด
# .median() → คำนวณค่ากลางของข้อมูล
c['age'] = c['age'].fillna(c['age'].median())

# phone_number: เติม '0' แล้วลบตัวอักษรที่ไม่ใช่ตัวเลขหรือ +
# r'[^0-9+]' → Regex หมายถึง 'ทุกอย่างที่ไม่ใช่' ตัวเลขหรือ +
# .replace(['','00'],'0') → ถ้าเหลือค่าว่างหรือ 00 ให้เป็น '0'
c['phone_number'] = (
    c['phone_number'].fillna('0').astype(str)
    .str.replace(r'[^0-9+]', '', regex=True)
    .replace(['', '00'], '0')
)

# promotion_id: เติม 0 แล้วแปลงเป็น int
# .astype(int) → แปลงชนิดข้อมูลจาก float → integer (ไม่มีทศนิยม)
s['promotion_id'] = s['promotion_id'].fillna(0).astype(int)

print('age missing    :', c['age'].isna().sum())
print('phone empty    :', (c['phone_number'] == '').sum())
print('promo missing  :', s['promotion_id'].isna().sum())

age missing    : 0
phone empty    : 0
promo missing  : 0


In [3]:
# --- 2. Date Conversion + Random Time 09:00–17:00 ---

def fix_dates(series):
    # pd.to_datetime() → แปลง string เป็น datetime
    # errors='coerce'  → ถ้าแปลงไม่ได้ให้เป็น NaT
    # format='mixed'   → รองรับหลายรูปแบบวันที่
    parsed = pd.to_datetime(series, errors='coerce', format='mixed')

    # .where(condition) → เก็บค่าที่เงื่อนไข True ไว้ ที่เหลือเปลี่ยนเป็น NaT
    # กรองปีเพี้ยนออก (ไม่อยู่ในช่วง 2000-2025)
    parsed = parsed.where(parsed.dt.year.between(2000, 2025))

    # .fillna() → เติม NaT ด้วยวันที่ default
    # .dt.normalize() → รีเซ็ตเวลาให้เป็น 00:00:00
    parsed = parsed.fillna(pd.Timestamp('2024-01-01')).dt.normalize()

    # np.random.randint() → สุ่มตัวเลขจำนวนเต็ม
    # 9*3600 = 32400 วินาที = 09:00 | 17*3600 = 61200 วินาที = 17:00
    rand_sec = np.random.randint(9*3600, 17*3600+1, size=len(parsed))

    # pd.to_timedelta() → แปลงวินาทีเป็น timedelta แล้วบวกเข้ากับวันที่
    return parsed + pd.to_timedelta(rand_sec, unit='s')

c['join_date']          = fix_dates(c['join_date'])
c['last_purchase_date'] = fix_dates(c['last_purchase_date'])
s['date']               = fix_dates(s['date'])

print('join_date dtype :', c['join_date'].dtype)
print('sales date dtype:', s['date'].dtype)
display(c[['join_date','last_purchase_date']].head(3))
display(s[['date']].head(3))

join_date dtype : datetime64[ns]
sales date dtype: datetime64[ns]


,join_date,last_purchase_date
0,2024-01-01 15:21:20,2024-01-01 12:56:21
1,2024-01-01 15:33:08,2024-01-01 16:06:59
2,2022-01-25 14:34:29,2024-01-01 09:24:33


,date
0,2023-11-30 15:24:43
1,2023-11-30 12:11:57
2,2023-11-30 16:46:43


In [4]:
# --- 3. Export ---
# .to_csv() → บันทึก DataFrame ออกเป็นไฟล์ CSV
# index=False → ไม่เอาเลขแถว (0,1,2,...) ใส่ในไฟล์
c.to_csv('customers_cleaned.csv', index=False)
s.to_csv('sales_transactions_cleaned.csv', index=False)
print('✅ Saved customers_cleaned.csv & sales_transactions_cleaned.csv')

# === จุดสังเกต ===
# ✔ c['age'].isna().sum() == 0
# ✔ c['join_date'].dtype == datetime64
# ✔ s['promotion_id'].dtype == int64
# ✔ ไฟล์ทั้งสองถูกสร้างขึ้นในโฟลเดอร์

✅ Saved customers_cleaned.csv & sales_transactions_cleaned.csv
